# PINN fundamentals

This notebook explains the small reusable framework in `src/pinn/core.py`. The framework is intentionally transparent: it shows the neural network, automatic differentiation, residual calculation, condition loss, and optimisation loop separately.

> **Learning goal:** explain why a network becomes a Physics-Informed Neural Network when a differential-equation residual is part of its loss.

## Conceptual workflow

```text
Coordinates → Neural network → Automatic differentiation → Physics residual
          ↘ known initial/boundary conditions ↗
                            ↓
                       Total loss → Optimiser
```


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Using project root: {PROJECT_ROOT}")


Using project root: /Users/nitinbhardwaj2006gmail.com/Desktop/physics-informed-neural-networks


## The trial solution

An MLP is used as a differentiable trial solution. For Newton cooling, it maps `t → y(t)`. For Burgers' equation, it maps `(x, t) → u(x, t)`. The architecture is configured through `MLPConfig`, so the core does not know the physical problem in advance.


In [2]:
from pinn.core import MLP, MLPConfig

model = MLP(MLPConfig(input_dim=1, output_dim=1, hidden_layers=(32, 32)))
print(model)


MLP(
  (network): Sequential(
    (0): Linear(in_features=1, out_features=32, bias=True)
    (1): Tanh()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): Tanh()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)


## Automatic differentiation

PyTorch can differentiate the network output with respect to input coordinates. The helper function below will later supply time and spatial derivatives to a physics residual. `create_graph=True` inside the helper matters because Burgers' equation needs a second derivative.


In [3]:
import torch
from pinn.core import derivative, second_derivative

x = torch.tensor([[1.0], [2.0]], requires_grad=True)
y = x**3

print("dy/dx:", derivative(y, x).ravel().tolist())
print("d2y/dx2:", second_derivative(y, x, component=0, coordinate=0).ravel().tolist())


dy/dx: [3.0, 12.0]
d2y/dx2: [6.0, 12.0]


## Residual loss and condition loss

The generic trainer receives two problem-specific functions: a residual function and a condition function. It minimizes the mean squared value of both. The next notebooks show how this idea is instantiated for cooling and Burgers' equation.

### Questions to answer

1. Why must the input tensor require gradients?
2. Why is a physics residual different from a labelled target value?
3. Why can initial and boundary conditions be necessary even if the residual is small?
